# Andrey Karpathy's MicroGrad
https://github.com/karpathy/micrograd

In [ ]:
import numpy as np
from graphviz import Digraph

In [ ]:
from graphviz.graphs import rendering
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad}, op={self._op})"

    def __add__(self, other):
        # Make sure other is an instance of Value
        other = other if isinstance(other, Value) else Value(other)

        # Create a value to store the result of the add operation
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            # Chain rule application
            print("\tself", self)
            print("\tother", other)
            print("\tout", out)
            self.grad = 1 * out.grad
            other.grad = 1 * out.grad
        out._backward = _backward

        return out

    def __mul__(self, other):
        # Make sure other is an instance of Value
        other = other if isinstance(other, Value) else Value(other)

        # Create a value to store the result of the mul operation
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            print("\tself", self)
            print("\tother", other)
            print("\tout", out)
            # Chain rule application
            self.grad = other.data * out.grad
            other.grad = self.data * out.grad
        out._backward = _backward

        return out

    def backward(self):
        # Depth-first search to build the topological order of the computational graph
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        # Start the topological sort from this node (usually the output node)
        build_topo(self)

        # Base case: ∂output/∂output = 1
        self.grad = 1

        # Iterate through the nodes in reverse topological order
        for node in reversed(topo):
            print(node)
            node._backward()

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 49)

In [ ]:
def forward(x, y, z):
    q = x + y
    f = q * z
    return f

In [ ]:
# Initialize input values
x = Value(-2)
y = Value(5)
z = Value(-4)

# Perform forward pass
f = forward(x, y, z)

# Perform backpropagation
print("\nBackpropagation:")
f.backward()

print("\nGradients:")
print(f"df/dx = {x.grad}")
print(f"df/dy = {y.grad}")
print(f"df/dz = {z.grad}")


Backpropagation:
Value(data=-12, grad=1, op=*)
	self Value(data=3, grad=0, op=+)
	other Value(data=-4, grad=0, op=)
	out Value(data=-12, grad=1, op=*)
Value(data=3, grad=-4, op=+)
	self Value(data=-2, grad=0, op=)
	other Value(data=5, grad=0, op=)
	out Value(data=3, grad=-4, op=+)
Value(data=5, grad=-4, op=)
Value(data=-2, grad=-4, op=)
Value(data=-4, grad=3, op=)

Gradients:
df/dx = -4
df/dy = -4
df/dz = 3
